# Amazon Polarity 100% Dataset — Head-Only Fine-Tuning

This notebook freezes the DistilBERT encoder and trains only the classification head.

## Notebook Overview

This notebook runs the **100% Amazon Polarity** experiment and saves results to:

`/content/drive/MyDrive/amazon-sentiment/summaries.json`

### Table of Contents
1. Install dependencies
2. Imports
3. Reproducibility and Google Drive setup
4. Load and prepare the 100% dataset
5. Tokenization and cached dataset splits
6. Shared metrics, memory, training, and saving utilities
7. Method-specific training experiment
8. Evaluation and summary saving

### Dataset Split Used Here
- Dataset: `fancyzhx/amazon_polarity`
- Validation holdout from training set: `50,000` rows
- Actual 100% training set used: all remaining training rows after validation holdout
- Evaluation during training: `5,000` validation rows for faster validation


## 1. Install Dependencies
Install or upgrade the libraries needed for Hugging Face training in Colab.


In [ ]:
# Install/upgrade the libraries used in the original notebook
!pip install -q -U transformers datasets accelerate peft scikit-learn torchao


## 2. Imports
Import dataset, modeling, metrics, training, LoRA, and utility libraries.


In [ ]:
import os
import json
import time
import math
import random
import inspect
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset, load_from_disk
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)
from transformers.utils.notebook import NotebookProgressCallback
from peft import LoraConfig, get_peft_model, TaskType


## 3. Reproducibility and Google Drive Setup
Set seeds, reset CUDA memory stats, mount Google Drive, and define the shared summary JSON path.


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/amazon-sentiment"
os.makedirs(SAVE_DIR, exist_ok=True)
SUMMARY_PATH = f"{SAVE_DIR}/summaries.json"

print("Seed set to:", SEED)
print("Summary JSON path:", SUMMARY_PATH)


## 4. Load and Prepare the 100% Dataset
Load Amazon Polarity, combine `title` and `content` into one text field, reserve validation rows, and define the full training split.


In [ ]:
# Load the same dataset used in the original notebook
DATASET_NAME = "fancyzhx/amazon_polarity"
dataset = load_dataset(DATASET_NAME)

print(dataset)
for split in dataset.keys():
    print(f"{split}: {len(dataset[split])}")

# Match the original text-combining step

def combine_text(example):
    title = example["title"].strip() if example["title"] else ""
    content = example["content"].strip() if example["content"] else ""
    example["text"] = (title + " " + content).strip()
    return example

dataset = dataset.map(combine_text)

train_dataset = dataset["train"].shuffle(seed=SEED)
test_dataset = dataset["test"]

VAL_SIZE = 50_000
val_full = train_dataset.select(range(VAL_SIZE))
train_full = train_dataset.select(range(VAL_SIZE, len(train_dataset)))
val_5k = val_full.select(range(5000))

print("train_full:", len(train_full))
print("val_full:", len(val_full))
print("val_5k:", len(val_5k))
print("test_full:", len(test_dataset))


## 5. Tokenization and Cached Splits
Load the DistilBERT tokenizer, tokenize the full training split and 5k validation split, and cache tokenized datasets on disk.


In [ ]:
MODEL_NAME = "distilbert-base-uncased"
NUM_LABELS = 2
MAX_LENGTH = 128

id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("Tokenizer loaded:", MODEL_NAME)
print("Tokenizer vocab size:", tokenizer.vocab_size)

def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )


def prepare_tokenized_split(split, cache_path=None):
    if cache_path and os.path.exists(cache_path):
        tokenized = load_from_disk(cache_path)
        print(f"Loaded cached tokenized split from {cache_path}")
        return tokenized

    tokenized = split.map(
        tokenize_function,
        batched=True,
        desc=f"Tokenizing {cache_path or 'dataset'}",
    )

    tokenized = tokenized.rename_column("label", "labels")
    keep_cols = ["input_ids", "attention_mask", "labels"]
    tokenized = tokenized.remove_columns(
        [col for col in tokenized.column_names if col not in keep_cols]
    )

    if cache_path:
        tokenized.save_to_disk(cache_path)
        print(f"Saved tokenized split to {cache_path}")

    return tokenized

TOKENIZED_FULL_PATH = "./tokenized_train_full_amazon_128"
TOKENIZED_VAL_PATH = "./tokenized_val_5k_amazon_128"

tokenized_train_full = prepare_tokenized_split(train_full, TOKENIZED_FULL_PATH)
tokenized_val_5k = prepare_tokenized_split(val_5k, TOKENIZED_VAL_PATH)

print(tokenized_train_full)
print(tokenized_val_5k)


## 6. Shared Utilities
Define metric computation, parameter counting, GPU memory tracking, training argument creation, and summary saving.


In [ ]:
def compute_classification_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }


def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    pct = 100 * trainable / total
    return {
        "total_params": total,
        "trainable_params": trainable,
        "trainable_pct": pct,
    }


def print_parameter_report(model, title="Model"):
    stats = count_parameters(model)
    print(f"{title} parameter report")
    print(f"  Total params     : {stats['total_params']:,}")
    print(f"  Trainable params : {stats['trainable_params']:,}")
    print(f"  Trainable %      : {stats['trainable_pct']:.4f}%")


def reset_gpu_memory():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def get_peak_gpu_memory_mb():
    if torch.cuda.is_available():
        return torch.cuda.max_memory_allocated() / (1024 ** 2)
    return None


def build_training_args(
    output_dir,
    learning_rate,
    epochs,
    per_device_train_batch_size,
    per_device_eval_batch_size=64,
    logging_steps=100,
):
    raw_args = dict(
        output_dir=output_dir,
        do_train=True,
        do_eval=True,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        save_total_limit=1,
        num_train_epochs=epochs,
        learning_rate=learning_rate,
        weight_decay=0.01,
        warmup_ratio=0.1,
        per_device_train_batch_size=per_device_train_batch_size,
        per_device_eval_batch_size=per_device_eval_batch_size,
        logging_strategy="steps",
        logging_steps=logging_steps,
        fp16=torch.cuda.is_available(),
        bf16=False,
        report_to="none",
        seed=SEED,
    )

    sig = inspect.signature(TrainingArguments.__init__)
    accepted = set(sig.parameters.keys())
    filtered_args = {k: v for k, v in raw_args.items() if k in accepted}
    return TrainingArguments(**filtered_args)


def save_summary(summary_key, summary_value):
    if os.path.exists(SUMMARY_PATH):
        with open(SUMMARY_PATH, "r") as f:
            summaries = json.load(f)
    else:
        summaries = {}

    summaries[summary_key] = summary_value

    with open(SUMMARY_PATH, "w") as f:
        json.dump(summaries, f, indent=2)

    print(f"saved: {summary_key}")
    print(f"Done. {len(summaries)} summaries now saved.")


## 6. Head-Only Model Builder
Freeze the DistilBERT encoder and keep only the classification head trainable.


In [ ]:
def build_head_only_model():
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        id2label=id2label,
        label2id=label2id,
    )

    for param in model.parameters():
        param.requires_grad = False

    for name, param in model.named_parameters():
        if name.startswith("pre_classifier") or name.startswith("classifier"):
            param.requires_grad = True

    return model


## 7. Build and Inspect the Head-Only Model
Create the model and print total/trainable parameter counts.


In [ ]:
head_only_full_model = build_head_only_model()
print_parameter_report(head_only_full_model, title="Head-only model for 100pct run")


## 8. Training Configuration
Set hyperparameters for the 100% head-only run.


In [ ]:
training_args_head_full = build_training_args(
    output_dir="./runs/head_only_full",
    learning_rate=5e-4,
    epochs=2,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    logging_steps=2000,
)
print(training_args_head_full)


## 9. Trainer Setup
Create the Hugging Face `Trainer` using the tokenized full training split and validation split.


In [ ]:
head_only_full_trainer = Trainer(
    model=head_only_full_model,
    args=training_args_head_full,
    train_dataset=tokenized_train_full,
    eval_dataset=tokenized_val_5k,
    data_collator=data_collator,
    compute_metrics=compute_classification_metrics,
)


## 10. Train the Head-Only Model
Remove the notebook progress callback, reset GPU memory stats, and run training.


In [ ]:
try:
    head_only_full_trainer.remove_callback(NotebookProgressCallback)
    print("NotebookProgressCallback removed")
except Exception as e:
    print("Callback removal note:", e)

reset_gpu_memory()
start_time = time.time()
head_only_full_trainer.train()
end_time = time.time()

head_only_full_train_time_sec = end_time - start_time
head_only_full_peak_mem_mb = get_peak_gpu_memory_mb()

print("Training finished.")
print(f"Wall-clock training time (sec): {head_only_full_train_time_sec:.2f}")
print(f"Peak GPU memory during training (MB): {head_only_full_peak_mem_mb:.2f}")


## 11. Evaluate and Create Summary
Evaluate on the 5k validation subset and store the key metrics in a summary dictionary.


In [ ]:
head_only_full_eval = head_only_full_trainer.evaluate()
print("Head-only 100pct eval results:")
for k, v in head_only_full_eval.items():
    print(f"{k}: {v}")

head_only_full_summary = {
    "method": "head_only_100pct",
    "train_samples": len(tokenized_train_full),
    "val_samples": len(tokenized_val_5k),
    "epochs": training_args_head_full.num_train_epochs,
    "trainable_params": count_parameters(head_only_full_model)["trainable_params"],
    "trainable_pct": count_parameters(head_only_full_model)["trainable_pct"],
    "train_time_sec": head_only_full_train_time_sec,
    "peak_gpu_mem_mb": head_only_full_peak_mem_mb,
    "eval_accuracy": head_only_full_eval.get("eval_accuracy"),
    "eval_f1_macro": head_only_full_eval.get("eval_f1_macro"),
    "eval_loss": head_only_full_eval.get("eval_loss"),
}

print(head_only_full_summary)


## 12. Save Head-Only Summary
Write the final 100% head-only summary into the shared JSON file.


In [ ]:
save_summary("head_only_100pct", head_only_full_summary)
